In [1]:
import polars as pl

from datetime import date

from ohlc_dss_model.data.data_loader import load_parquet
from ohlc_dss_model.data.integrity import sort_data, remove_incomplete_days
from ohlc_dss_model.data.tagging import intraday_session_tagging, session_tagging

from ohlc_dss_model.features.session_aggregation import aggregate_sessions

from ohlc_dss_model.utils.dt_utils import convert_to_timezone


In [2]:
df = load_parquet()
df = convert_to_timezone(df)
df = sort_data(df)
df = session_tagging(df)
df = intraday_session_tagging(df)
df = remove_incomplete_days(df)
df = df.filter(
    (pl.col("Session") != date(2016, 11, 15)), (pl.col("Session") != date(2025, 10, 1))
)
print(df.head(5))

shape: (5, 9)
┌────────────────┬────────┬────────┬────────┬───┬────────┬────────────┬────────────┬───────────────┐
│ DateTime       ┆ Open   ┆ High   ┆ Low    ┆ … ┆ Volume ┆ TickVolume ┆ Session    ┆ Intraday_Sess │
│ ---            ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---    ┆ ---        ┆ ---        ┆ ion           │
│ datetime[μs,   ┆ f64    ┆ f64    ┆ f64    ┆   ┆ i64    ┆ i64        ┆ date       ┆ ---           │
│ America/New_Yo ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆ str           │
│ rk]            ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆               │
╞════════════════╪════════╪════════╪════════╪═══╪════════╪════════════╪════════════╪═══════════════╡
│ 2016-11-15     ┆ 4765.7 ┆ 4770.5 ┆ 4765.7 ┆ … ┆ 0      ┆ 262        ┆ 2016-11-16 ┆ Asia          │
│ 18:00:00 EST   ┆        ┆        ┆        ┆   ┆        ┆            ┆            ┆               │
│ 2016-11-15     ┆ 4769.9 ┆ 4772.7 ┆ 4768.6 ┆ … ┆ 0      ┆ 259        ┆ 2016-

We need to group our data by its session such that each row represents one trading day

We will not feed the model the raw intraday data, instead we will summarize those into features.

In [3]:
session_df = (
    df.group_by(["Session", "Intraday_Session"])
    .agg([
        pl.col("Open").first().alias("O"),
        pl.col("High").max().alias("H"),
        pl.col("Low").min().alias("L"),
        pl.col("Close").last().alias("C"),
    ])
    
    .pivot(
        index="Session",
        on="Intraday_Session",
        values=["O", "H", "L", "C"],
    )
    .sort("Session")
)
print(session_df.head(5))
print(session_df.shape)
print(session_df.columns)


shape: (5, 13)
┌────────────┬──────────┬────────────┬────────┬───┬────────┬──────────┬────────────┬────────┐
│ Session    ┆ O_London ┆ O_New York ┆ O_Asia ┆ … ┆ L_Asia ┆ C_London ┆ C_New York ┆ C_Asia │
│ ---        ┆ ---      ┆ ---        ┆ ---    ┆   ┆ ---    ┆ ---      ┆ ---        ┆ ---    │
│ date       ┆ f64      ┆ f64        ┆ f64    ┆   ┆ f64    ┆ f64      ┆ f64        ┆ f64    │
╞════════════╪══════════╪════════════╪════════╪═══╪════════╪══════════╪════════════╪════════╡
│ 2016-11-16 ┆ 4762.2   ┆ 4747.5     ┆ 4765.7 ┆ … ┆ 4762.0 ┆ 4747.5   ┆ 4785.9     ┆ 4762.0 │
│ 2016-11-17 ┆ 4793.7   ┆ 4790.7     ┆ 4778.8 ┆ … ┆ 4778.8 ┆ 4790.7   ┆ 4831.7     ┆ 4793.9 │
│ 2016-11-18 ┆ 4834.2   ┆ 4830.4     ┆ 4830.2 ┆ … ┆ 4823.7 ┆ 4830.7   ┆ 4812.4     ┆ 4834.4 │
│ 2016-11-21 ┆ 4820.5   ┆ 4823.8     ┆ 4806.4 ┆ … ┆ 4805.5 ┆ 4823.8   ┆ 4858.8     ┆ 4820.8 │
│ 2016-11-22 ┆ 4881.0   ┆ 4875.1     ┆ 4858.0 ┆ … ┆ 4858.0 ┆ 4875.3   ┆ 4879.8     ┆ 4881.0 │
└────────────┴──────────┴────────────┴───────

In [4]:
features = aggregate_sessions(df)
print(features.head(5))

shape: (5, 13)
┌────────────┬────────────┬────────┬──────────┬───┬──────────┬────────────┬────────┬──────────┐
│ Session    ┆ O_New York ┆ O_Asia ┆ O_London ┆ … ┆ L_London ┆ C_New York ┆ C_Asia ┆ C_London │
│ ---        ┆ ---        ┆ ---    ┆ ---      ┆   ┆ ---      ┆ ---        ┆ ---    ┆ ---      │
│ date       ┆ f64        ┆ f64    ┆ f64      ┆   ┆ f64      ┆ f64        ┆ f64    ┆ f64      │
╞════════════╪════════════╪════════╪══════════╪═══╪══════════╪════════════╪════════╪══════════╡
│ 2016-11-16 ┆ 4747.5     ┆ 4765.7 ┆ 4762.2   ┆ … ┆ 4745.5   ┆ 4785.9     ┆ 4762.0 ┆ 4747.5   │
│ 2016-11-17 ┆ 4790.7     ┆ 4778.8 ┆ 4793.7   ┆ … ┆ 4790.2   ┆ 4831.7     ┆ 4793.9 ┆ 4790.7   │
│ 2016-11-18 ┆ 4830.4     ┆ 4830.2 ┆ 4834.2   ┆ … ┆ 4817.9   ┆ 4812.4     ┆ 4834.4 ┆ 4830.7   │
│ 2016-11-21 ┆ 4823.8     ┆ 4806.4 ┆ 4820.5   ┆ … ┆ 4805.8   ┆ 4858.8     ┆ 4820.8 ┆ 4823.8   │
│ 2016-11-22 ┆ 4875.1     ┆ 4858.0 ┆ 4881.0   ┆ … ┆ 4873.8   ┆ 4879.8     ┆ 4881.0 ┆ 4875.3   │
└────────────┴───────────